In [1]:
import polars as pl

train = pl.read_csv('/kaggle/input/nvidia-nemotron-3-reasoning-challenge/train.csv')

train.head()

id,prompt,answer
str,str,str
"""00066667""","""In Alice's Wonderland, a secre…","""10010111"""
"""000b53cf""","""In Alice's Wonderland, a secre…","""01000011"""
"""00189f6a""","""In Alice's Wonderland, secret …","""cat imagines book"""
"""001b24c4""","""In Alice's Wonderland, numbers…","""XXXVIII"""
"""001c63cb""","""In Alice's Wonderland, secret …","""wizard creates secret"""


In [2]:
!nvidia-smi

Wed May  6 16:07:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   31C    P0             46W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [3]:
!ls /kaggle/input/datasets/lkk688/

cleandata  reasoning-sft-combined  unslothpackages


In [6]:
!ls /kaggle/input/datasets/lkk688/reasoning-sft-combined

sft_combined_v5.jsonl


In [41]:
%%writefile nemotron_unsloth_mixed_sft_v3_2.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Nemotron 3 High-Performance LoRA SFT Trainer
Supports Unsloth (fast path) with automatic fallback to vanilla HF + PEFT.

Accepts one or more JSONL files via --reasoning_jsonl (space-separated).
Each JSONL file may use the plain schema {id, prompt, reasoning, answer, task_type, source}
OR the extended v3 schema that also includes {quality_score, needs_downweight, challenge_native, ...}.

Non-challenge / distill / regularize samples with needs_downweight=True are sub-sampled
at --downweight_factor (default 1.0 = keep all, 0.3 = keep 30%).
"""

import os
import sys
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
# HF_HUB_ENABLE_HF_TRANSFER is only useful with internet; skipped by default so
# Kaggle offline mode works without change.

import argparse
import json
import re
import random
import shutil
import subprocess
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Union

import torch

# ---------------------------------------------------------------------------
# Optional Unsloth import — gracefully fall back to vanilla HF + PEFT
# ---------------------------------------------------------------------------
USE_UNSLOTH = False
try:
    from unsloth import FastLanguageModel, is_bfloat16_supported
    USE_UNSLOTH = True
    print("[*] Unsloth found — using optimised fast path.")
except ImportError:
    print("[*] Unsloth not found — using vanilla HF + PEFT path.")

# ---------------------------------------------------------------------------
# Optional bitsandbytes — needed for 4-bit quant and adamw_8bit
# ---------------------------------------------------------------------------
HAS_BNB = False
try:
    import bitsandbytes  # noqa: F401
    HAS_BNB = True
except ImportError:
    pass

from datasets import Dataset, Features, Sequence, Value
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Trainer,
    TrainingArguments,
)
import transformers as _transformers_mod
# transformers ≥ 5.0 deprecates torch_dtype= in favour of dtype=
_USE_DTYPE_KWARG = tuple(int(x) for x in _transformers_mod.__version__.split(".")[:2]) >= (5, 0)

if not USE_UNSLOTH:
    from peft import LoraConfig, TaskType, get_peft_model


# ---------------------------------------------------------------------------
# Data structures
# ---------------------------------------------------------------------------

@dataclass
class Sample:
    source: str
    messages: List[Dict[str, str]]
    meta: Dict[str, Any]
    needs_downweight: bool = False


# ---------------------------------------------------------------------------
# Arguments
# ---------------------------------------------------------------------------

def parse_args():
    p = argparse.ArgumentParser(description="Nemotron SFT trainer (Unsloth optional, multi-source)")
    p.add_argument("--model_name", type=str, required=True)
    p.add_argument("--output_dir", type=str, required=True)
    p.add_argument("--reasoning_jsonl", type=str, nargs="+", required=True,
                   help="One or more JSONL files. Accepts train_reasoning, sft_combined, "
                        "distill, aug, regularize outputs. Space-separated.")

    # Data quality / mixing
    p.add_argument("--min_quality_score", type=float, default=0.0,
                   help="Reject records with quality_score below this (0=no filter). "
                        "Only applies to records that have quality_score field.")
    p.add_argument("--max_reasoning_words", type=int, default=600,
                   help="Hard cap on reasoning word count; 0 = disabled (default 600). "
                        "Drops verbose outliers (e.g. raw Qwen3 CoT chains >2000 words).")
    p.add_argument("--downweight_factor", type=float, default=0.25,
                   help="Fraction of needs_downweight=True samples to keep (1.0=all, 0.25=25%%). "
                        "Default 0.25 keeps math/distill data from dominating the challenge signal.")

    # Formatting
    p.add_argument("--use_chat_template", action="store_true", default=True)
    p.add_argument("--boxed_answer", action="store_true",
                   help="Wrap final answers in \\boxed{}")
    p.add_argument("--include_think_for_reasoning", action="store_true", default=True)
    p.add_argument("--system_prompt", type=str,
                   default="You are an expert logical reasoning assistant. "
                           "Always output your thought process in <think> tags before providing the final answer.")

    # Training
    p.add_argument("--max_seq_length", type=int, default=4096)
    p.add_argument("--num_train_epochs", type=float, default=2.0)
    p.add_argument("--per_device_train_batch_size", type=int, default=4)
    p.add_argument("--gradient_accumulation_steps", type=int, default=4)
    p.add_argument("--learning_rate", type=float, default=1e-4)
    p.add_argument("--weight_decay", type=float, default=0.01)
    p.add_argument("--warmup_steps", type=int, default=100)
    p.add_argument("--logging_steps", type=int, default=5)
    p.add_argument("--save_steps", type=int, default=200)
    p.add_argument("--save_total_limit", type=int, default=1,
                   help="Keep only the N most recent checkpoints (default 1). "
                        "Prevents disk exhaustion on Kaggle's 20GB working dir.")
    p.add_argument("--save_only_model", action="store_true",
                   help="Save only adapter weights, skip optimizer/scheduler states. "
                        "Saves ~1.5-4 GB per checkpoint but disables --resume_from_checkpoint.")
    p.add_argument("--resume_from_checkpoint", type=str, default="",
                   help="Path to a checkpoint dir (e.g. output_dir/checkpoint-200) to resume from.")
    p.add_argument("--seed", type=int, default=3407)
    p.add_argument("--dataloader_num_workers", type=int, default=4,
                   help="DataLoader worker processes (default 4)")
    p.add_argument("--group_by_length", action="store_true",
                   help="Group samples by length to minimise padding waste")

    # Performance / hardware
    p.add_argument("--flash_attn", action="store_true",
                   help="Use Flash Attention 2 (attn_implementation='flash_attention_2'). "
                        "Recommended for Blackwell/Ampere GPUs — gives 2-4x speedup.")
    p.add_argument("--no_gradient_checkpointing", action="store_true",
                   help="Disable gradient checkpointing. Safe when VRAM is sufficient "
                        "(e.g. 96GB RTX PRO 6000 + LoRA-only gradients). Speeds up ~20%%.")
    p.add_argument("--tf32", action="store_true", default=True,
                   help="Enable TF32 matmuls (default on). Blackwell/Ampere native — free throughput.")
    p.add_argument("--torch_compile", action="store_true",
                   help="Wrap model with torch.compile (experimental, ~10-20%% speedup after warmup).")

    # Model / LoRA
    p.add_argument("--load_in_4bit", action="store_true")
    p.add_argument("--dtype", type=str, default="bf16", choices=["auto", "bf16", "fp16"])
    p.add_argument("--lora_r", type=int, default=32)
    p.add_argument("--lora_alpha", type=int, default=64)
    p.add_argument("--lora_dropout", type=float, default=0.0)
    p.add_argument("--target_preset", type=str, default="kaggle",
                   choices=["kaggle", "kaggle_deep_only", "all_linear"])

    # Force standard HF path even when Unsloth is installed
    p.add_argument("--no_unsloth", action="store_true",
                   help="Disable Unsloth even if installed")

    # ── Kaggle / offline environment ──────────────────────────────────────────
    p.add_argument("--kaggle_mode", action="store_true",
                   help="Enable Kaggle offline setup: mount mamba_ssm path, copy Triton, "
                        "optionally install local packages. Should be set before --model_name "
                        "is resolved.")
    p.add_argument("--cutlass_path", type=str,
                   default="/kaggle/usr/lib/notebooks/ryanholbrook/"
                           "nvidia-utility-script/nvidia_cutlass_dsl/python_packages/",
                   help="site.addsitedir() path that exposes mamba_ssm (Kaggle only)")
    p.add_argument("--triton_src", type=str,
                   default="/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/triton",
                   help="Read-only Triton source dir to copy (Kaggle only)")
    p.add_argument("--triton_dst", type=str,
                   default="/kaggle/working/triton",
                   help="Writable destination for the Triton copy")
    p.add_argument("--local_packages_dir", type=str, default="",
                   help="Directory with offline .whl files; runs pip install --no-index "
                        "--find-links=<dir> before training (for Kaggle no-internet)")
    p.add_argument("--model_kaggle_id", type=str, default="",
                   help="Kaggle model ID passed to kagglehub.model_download(), e.g. "
                        "'nvidia/nemotron-h/transformers/nemotron-h-8b-bf16'. "
                        "When set, overrides --model_name with the downloaded local path.")

    return p.parse_args()


# ---------------------------------------------------------------------------
# Utilities
# ---------------------------------------------------------------------------

def seed_everything(seed: int, tf32: bool = True):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if tf32:
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True


def resolve_dtype(name: str):
    if name == "bf16":
        return torch.bfloat16
    if name == "fp16":
        return torch.float16
    return None


def ensure_pad_token(tokenizer):
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token if tokenizer.eos_token_id is not None else "<|pad|>"


def build_target_modules(preset: str) -> Union[List[str], str]:
    preset = preset.strip().lower()
    if preset == "kaggle":
        return ["q_proj", "k_proj", "v_proj", "o_proj",
                "gate_proj", "up_proj", "down_proj",
                "in_proj", "out_proj", "x_proj", "dt_proj"]
    if preset == "kaggle_deep_only":
        return [r".*layers\.[3-6][0-9]\..*(proj)"]
    if preset == "all_linear":
        return "all-linear"
    return ["in_proj", "out_proj", "up_proj", "down_proj"]


# ---------------------------------------------------------------------------
# Kaggle offline environment setup
# ---------------------------------------------------------------------------

def setup_kaggle_env(args) -> None:
    """
    Mount the Kaggle NVIDIA utility-script path (exposes mamba_ssm), copy Triton
    to a writable directory so its ptxas binary is executable, and optionally
    pip-install packages from a local wheel directory.

    Must be called BEFORE model loading so mamba_ssm is importable when
    transformers initialises the Nemotron architecture.
    """
    import site as _site

    # 1. Add the nvidia-utility-script root (contains flash_attn, mamba_ssm, etc.)
    #    cutlass_path is  .../nvidia-utility-script/nvidia_cutlass_dsl/python_packages/
    #    so parent.parent is the root that also holds flash_attn_2_cuda.so
    nvidia_script_root = str(Path(args.cutlass_path).parent.parent)
    if os.path.isdir(nvidia_script_root):
        _site.addsitedir(nvidia_script_root)
        print(f"[Kaggle] nvidia-utility-script root added: {nvidia_script_root}")
    # Also add the cutlass DSL sub-path (mamba_ssm lives here)
    if os.path.isdir(args.cutlass_path):
        _site.addsitedir(args.cutlass_path)
        print(f"[Kaggle] cutlass DSL path added: {args.cutlass_path}")
    else:
        print(f"[Kaggle][WARN] cutlass_path not found: {args.cutlass_path}")

    # 2. Triton ptxas fix — copy the small ptxas binaries to a dedicated writable dir.
    #
    # Why this approach instead of copying the whole triton package:
    #   • Kaggle adds /kaggle/usr/lib/.../nvidia-utility-script to sys.path at kernel
    #     startup with high priority, so triton is always imported from the read-only path
    #     regardless of sys.path inserts done later in this script.
    #   • But triton's knobs.py reads ptxas path from env vars before executing anything.
    #     Setting the env vars to a writable binary is therefore sufficient and reliable.
    #   • Two separate env vars exist: TRITON_PTXAS_PATH (arch < 100) and
    #     TRITON_PTXAS_BLACKWELL_PATH (arch >= 100, i.e. RTX PRO 6000 / B100).
    #     The original code only set TRITON_PTXAS_PATH, which is why Blackwell GPUs
    #     fell back to the read-only binary and hit PermissionError.
    ptxas_bin_dir = os.path.join(str(Path(args.triton_dst).parent), "bin")  # /kaggle/working/bin
    os.makedirs(ptxas_bin_dir, exist_ok=True)
    triton_bin_ro = os.path.join(args.triton_src, "backends", "nvidia", "bin")
    for binary, env_var in [
        ("ptxas-blackwell", "TRITON_PTXAS_BLACKWELL_PATH"),
        ("ptxas",           "TRITON_PTXAS_PATH"),
    ]:
        src_bin = os.path.join(triton_bin_ro, binary)
        dst_bin = os.path.join(ptxas_bin_dir, binary)
        if os.path.exists(src_bin):
            if not os.path.exists(dst_bin):
                shutil.copy2(src_bin, dst_bin)
                os.chmod(dst_bin, 0o755)
                print(f"[Kaggle] ptxas binary → {dst_bin}")
            os.environ[env_var] = dst_bin
            print(f"[Kaggle] {env_var}={dst_bin}")
        else:
            print(f"[Kaggle][WARN] ptxas binary not found: {src_bin}")
    # Put the bin dir on PATH too so triton subprocess calls find the right binary
    os.environ["PATH"] = ptxas_bin_dir + ":" + os.environ.get("PATH", "")

    # Also copy the full triton tree for cases where sys.path ordering works out
    # (lightweight kernels or fresh sessions where nvidia-utility-script isn't pre-loaded).
    # This is ~640 MB but optional — the env vars above are the real fix.
    if os.path.isdir(args.triton_src):
        if not os.path.exists(args.triton_dst):
            print(f"[Kaggle] Copying Triton → {args.triton_dst} ...")
            shutil.copytree(args.triton_src, args.triton_dst)
            os.system(f"chmod -R 777 {args.triton_dst}")
            print("[Kaggle] Triton copy done.")
        else:
            print(f"[Kaggle] Triton already at {args.triton_dst}")
    else:
        print(f"[Kaggle][WARN] triton_src not found: {args.triton_src}")

    # Add writable triton to the FRONT of sys.path; also invalidate any cached
    # read-only triton import so the writable copy is used on first real import.
    triton_parent = str(Path(args.triton_dst).parent)  # /kaggle/working
    if triton_parent not in sys.path:
        sys.path.insert(0, triton_parent)
    if os.path.isdir(args.triton_dst) and "triton" in sys.modules:
        triton_file = getattr(sys.modules["triton"], "__file__", "")
        if triton_file and not triton_file.startswith(triton_parent):
            print(f"[Kaggle] Evicting read-only triton from sys.modules ({triton_file[:60]})")
            for key in list(sys.modules):
                if key == "triton" or key.startswith("triton."):
                    del sys.modules[key]

    # 3. Local packages: install from offline wheels if provided
    if args.local_packages_dir and os.path.isdir(args.local_packages_dir):
        pkg_dir = args.local_packages_dir

        # Install unsloth from source if present
        unsloth_src = os.path.join(pkg_dir, "unsloth_source")
        if os.path.isdir(unsloth_src):
            print(f"[Kaggle] Installing Unsloth from source: {unsloth_src}")
            subprocess.run([sys.executable, "-m", "pip", "install", unsloth_src],
                           check=False, capture_output=False)

        # Install remaining wheels
        print(f"[Kaggle] Installing packages from {pkg_dir} (no-index) ...")
        result = subprocess.run(
            [sys.executable, "-m", "pip", "install",
             "--no-index", f"--find-links={pkg_dir}",
             "trl", "peft", "accelerate", "bitsandbytes", "datasets"],
            capture_output=True, text=True,
        )
        if result.returncode == 0:
            print("[Kaggle] Local packages installed.")
        else:
            # non-fatal: some packages may already be present
            print(f"[Kaggle][WARN] pip install exited {result.returncode}: "
                  f"{result.stderr[:300]}")


def resolve_model_path(args) -> str:
    """
    Return the local model path.
    If --model_kaggle_id is set, download via kagglehub and return the cache path.
    Otherwise return --model_name as-is (local path or HF hub id with internet).
    """
    if args.model_kaggle_id:
        try:
            import kagglehub
            print(f"[*] Downloading model via kagglehub: {args.model_kaggle_id}")
            path = kagglehub.model_download(args.model_kaggle_id)
            print(f"[*] Model cached at: {path}")
            return path
        except ImportError:
            print("[WARN] kagglehub not installed — falling back to --model_name")
        except Exception as exc:
            print(f"[WARN] kagglehub.model_download failed ({exc}) — falling back to --model_name")
    return args.model_name


# ---------------------------------------------------------------------------
# Model loading — Unsloth vs vanilla HF
# ---------------------------------------------------------------------------

def load_model_unsloth(args, model_path: str):
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_path,
        max_seq_length=args.max_seq_length,
        dtype=resolve_dtype(args.dtype),
        load_in_4bit=args.load_in_4bit,
        trust_remote_code=True,
    )
    ensure_pad_token(tokenizer)

    target_modules = build_target_modules(args.target_preset)
    print(f"[*] Applying LoRA (Unsloth) preset={args.target_preset}")
    model = FastLanguageModel.get_peft_model(
        model,
        r=args.lora_r,
        target_modules=target_modules,
        lora_alpha=args.lora_alpha,
        lora_dropout=args.lora_dropout,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=args.seed,
    )
    return model, tokenizer


def load_model_hf(args, model_path: str):
    dtype = resolve_dtype(args.dtype)
    quant_kwargs = {}

    if args.load_in_4bit:
        if not HAS_BNB:
            print("[!] bitsandbytes not found — load_in_4bit disabled, loading in full precision.")
        else:
            from transformers import BitsAndBytesConfig
            quant_kwargs["quantization_config"] = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_compute_dtype=dtype or torch.bfloat16,
                bnb_4bit_use_double_quant=True,
                bnb_4bit_quant_type="nf4",
            )

    # transformers ≥5.0 uses dtype=; older versions use torch_dtype=
    dtype_kw = "dtype" if _USE_DTYPE_KWARG else "torch_dtype"
    attn_kwargs = {}
    if getattr(args, "flash_attn", False):
        attn_kwargs["attn_implementation"] = "flash_attention_2"
        print("[*] Flash Attention 2 requested.")

    def _load_model(extra_kwargs):
        return AutoModelForCausalLM.from_pretrained(
            model_path,
            **{dtype_kw: dtype},
            device_map="auto",
            trust_remote_code=True,
            **extra_kwargs,
            **quant_kwargs,
        )

    try:
        model = _load_model(attn_kwargs)
        if attn_kwargs:
            print("[*] Flash Attention 2 enabled.")
    except ValueError as exc:
        if attn_kwargs and ("flash" in str(exc).lower() or "attn_implementation" in str(exc).lower()):
            # Model architecture (e.g. Nemotron-H Mamba-hybrid) doesn't support FA2 —
            # Mamba layers have no standard attention, so FA2 is simply not applicable.
            print(f"[WARN] Flash Attention 2 not supported by this model — falling back to default attention.\n"
                  f"       ({exc})")
            model = _load_model({})
        else:
            raise
    tokenizer = AutoTokenizer.from_pretrained(
        model_path, trust_remote_code=True
    )
    ensure_pad_token(tokenizer)

    target_modules = build_target_modules(args.target_preset)
    print(f"[*] Applying LoRA (PEFT) preset={args.target_preset}")
    lora_cfg = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=args.lora_r,
        lora_alpha=args.lora_alpha,
        lora_dropout=args.lora_dropout,
        target_modules=target_modules,
        bias="none",
        inference_mode=False,
    )
    model = get_peft_model(model, lora_cfg)
    # Required for gradient checkpointing + LoRA: tells PyTorch that the frozen
    # base weights still need gradients to flow through them (they don't store
    # grads themselves, but activations must be retained for the LoRA layers).
    model.enable_input_require_grads()
    model.print_trainable_parameters()
    return model, tokenizer


# ---------------------------------------------------------------------------
# Dataset construction
# ---------------------------------------------------------------------------

def _build_assistant_content(reasoning: str, answer: str, include_think: bool, boxed_answer: bool) -> str:
    if boxed_answer and answer and "\\boxed{" not in answer:
        answer = f"\\boxed{{{answer}}}"
    if reasoning:
        return (
            f"<think>\n{reasoning}\n</think>\n{answer}"
            if include_think
            else f"{reasoning}\n\n{answer}"
        )
    return answer


def load_reasoning_jsonl(
    paths: List[str],
    include_think: bool,
    boxed_answer: bool,
    system_prompt: str,
    min_quality_score: float,
    max_reasoning_words: int,
    downweight_factor: float,
    rng: random.Random,
) -> List[Sample]:
    """Load one or more JSONL files and apply quality/downweight filters."""
    all_samples: List[Sample] = []
    task_ct: Counter = Counter()
    source_ct: Counter = Counter()
    filtered_quality = 0
    filtered_words = 0
    filtered_downweight = 0

    for path in paths:
        p = Path(path)
        if not p.exists():
            print(f"[!] File not found, skipping: {path}")
            continue

        file_samples: List[Sample] = []
        with open(p, "r", encoding="utf-8") as f:
            for line in f:
                if not line.strip():
                    continue
                obj = json.loads(line.strip())

                # Quality score filter (only when field is present)
                if "quality_score" in obj and min_quality_score > 0:
                    qs = float(obj["quality_score"] or 0)
                    if qs < min_quality_score:
                        filtered_quality += 1
                        continue

                # Reasoning word-count cap
                if max_reasoning_words > 0:
                    reasoning_text = obj.get("reasoning", "") or ""
                    wc = obj.get("reasoning_word_count") or len(re.findall(r"\S+", reasoning_text))
                    if wc > max_reasoning_words:
                        filtered_words += 1
                        continue

                # Downweight subsampling
                needs_dw = bool(obj.get("needs_downweight", False))
                if needs_dw and downweight_factor < 1.0:
                    if rng.random() > downweight_factor:
                        filtered_downweight += 1
                        continue

                prompt = obj.get("prompt", "")
                reasoning = obj.get("reasoning", "").strip()
                answer = obj.get("answer", "").strip()
                task_type = obj.get("task_type", "unknown")
                source = obj.get("source", p.stem)

                if not prompt or not answer:
                    continue

                assistant_content = _build_assistant_content(
                    reasoning, answer, include_think, boxed_answer
                )

                messages: List[Dict[str, str]] = []
                if system_prompt:
                    messages.append({"role": "system", "content": system_prompt})
                messages.append({"role": "user", "content": prompt})
                messages.append({"role": "assistant", "content": assistant_content})

                file_samples.append(Sample(
                    source=source,
                    messages=messages,
                    meta={"id": obj.get("id"), "task_type": task_type},
                    needs_downweight=needs_dw,
                ))
                task_ct[task_type] += 1
                source_ct[source] += 1

        print(f"[*]   {p.name}: {len(file_samples)} samples kept")
        all_samples.extend(file_samples)

    print(f"\n[*] Total samples loaded: {len(all_samples)}")
    if filtered_quality:
        print(f"    Filtered by quality_score<{min_quality_score}: {filtered_quality}")
    if filtered_words:
        print(f"    Filtered by reasoning_words>{max_reasoning_words}: {filtered_words}")
    if filtered_downweight:
        print(f"    Sub-sampled (downweight factor={downweight_factor}): {filtered_downweight}")

    print("\n    By task type:")
    for t, c in sorted(task_ct.items()):
        print(f"      {t:25s}: {c:5d}")
    print("\n    By source (top 10):")
    for s, c in source_ct.most_common(10):
        print(f"      {s:45s}: {c:5d}")

    return all_samples


def _template_to_ids(result) -> List[int]:
    """Normalise apply_chat_template output to a plain Python list of ints.

    transformers ≥5.x returns tokenizers.Encoding (has .ids) or BatchEncoding
    (has .input_ids); older versions return a plain list.
    """
    if hasattr(result, "ids"):          # tokenizers.Encoding
        return [int(x) for x in result.ids]
    if hasattr(result, "input_ids"):    # BatchEncoding
        ids = result.input_ids
        return [int(x) for x in (ids[0] if hasattr(ids, "__getitem__") and not isinstance(ids[0], int) else ids)]
    return [int(x) for x in result]    # plain list


def render_messages_with_mask(tokenizer, messages: List[Dict[str, str]]) -> Dict[str, List[int]]:
    last_asst = max(
        (i for i, m in enumerate(messages) if m.get("role") == "assistant"),
        default=-1,
    )
    prefix = messages[:last_asst]
    suffix = [messages[last_asst]]

    try:
        prefix_ids = _template_to_ids(
            tokenizer.apply_chat_template(prefix, tokenize=True, add_generation_prompt=False)
        )
        full_ids = _template_to_ids(
            tokenizer.apply_chat_template(prefix + suffix, tokenize=True, add_generation_prompt=False)
        )
        labels = [-100] * len(prefix_ids) + full_ids[len(prefix_ids):]
        return {"input_ids": full_ids, "labels": labels}
    except Exception as e:
        raise RuntimeError(f"Chat template rendering failed: {e}")


def tokenize_samples(samples: List[Sample], tokenizer, max_seq_length: int) -> Dataset:
    rows: Dict[str, list] = {"input_ids": [], "attention_mask": [], "labels": []}
    skipped = 0
    for s in samples:
        try:
            enc = render_messages_with_mask(tokenizer, s.messages)
        except RuntimeError:
            skipped += 1
            continue
        input_ids = enc["input_ids"][:max_seq_length]
        labels = enc["labels"][:max_seq_length]
        rows["input_ids"].append(input_ids)
        rows["attention_mask"].append([1] * len(input_ids))
        rows["labels"].append(labels)

    if skipped:
        print(f"[!] Skipped {skipped} samples due to tokenisation errors.")
    print(f"[*] Final tokenised dataset size: {len(rows['input_ids'])} samples")
    # Explicit features avoid PyArrow overflow on ragged variable-length sequences
    # and work around datasets inferring the wrong type for tokenizers.Encoding objects.
    features = Features({
        "input_ids":      Sequence(Value("int32")),
        "attention_mask": Sequence(Value("int32")),
        "labels":         Sequence(Value("int32")),
    })
    return Dataset.from_dict(rows, features=features)


# ---------------------------------------------------------------------------
# Optimizer selection
# ---------------------------------------------------------------------------

def pick_optimizer(dtype: str) -> str:
    if HAS_BNB:
        return "adamw_8bit"
    if dtype == "bf16":
        return "adamw_torch_fused"
    return "adamw_torch"


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------

def main():
    args = parse_args()
    os.makedirs(args.output_dir, exist_ok=True)
    seed_everything(args.seed, tf32=args.tf32)
    rng = random.Random(args.seed)

    # ── Kaggle offline env: must run before model loading (mamba_ssm needs the path)
    if args.kaggle_mode:
        print("[*] Kaggle mode: setting up offline environment ...")
        setup_kaggle_env(args)

    model_path = resolve_model_path(args)

    use_unsloth = USE_UNSLOTH and not args.no_unsloth

    if use_unsloth:
        print("[*] Loading model with Unsloth ...")
        model, tokenizer = load_model_unsloth(args, model_path)
    else:
        print("[*] Loading model with vanilla HF + PEFT ...")
        model, tokenizer = load_model_hf(args, model_path)

    print(f"\n[*] Loading dataset from {len(args.reasoning_jsonl)} file(s):")
    for f in args.reasoning_jsonl:
        print(f"      {f}")

    samples = load_reasoning_jsonl(
        paths=args.reasoning_jsonl,
        include_think=args.include_think_for_reasoning,
        boxed_answer=args.boxed_answer,
        system_prompt=args.system_prompt,
        min_quality_score=args.min_quality_score,
        max_reasoning_words=args.max_reasoning_words,
        downweight_factor=args.downweight_factor,
        rng=rng,
    )

    if not samples:
        raise RuntimeError("No samples loaded — check your --reasoning_jsonl paths.")

    # Global shuffle ensures challenge / distill / augment samples are interleaved
    random.shuffle(samples)

    print("\n[*] Tokenizing dataset ...")
    train_ds = tokenize_samples(samples, tokenizer, args.max_seq_length)

    collator = DataCollatorForSeq2Seq(
        tokenizer=tokenizer,
        padding=True,
        label_pad_token_id=-100,
        return_tensors="pt",
    )

    optim = pick_optimizer(args.dtype)
    print(f"[*] Using optimizer: {optim}")

    use_gc = (not use_unsloth) and (not args.no_gradient_checkpointing)
    if args.no_gradient_checkpointing:
        print("[*] Gradient checkpointing disabled (--no_gradient_checkpointing).")

    import inspect as _inspect
    _valid_train_params = set(_inspect.signature(TrainingArguments.__init__).parameters)

    def _train_arg(name, value):
        """Return {name: value} only if this transformers version accepts it."""
        if name in _valid_train_params:
            return {name: value}
        print(f"[WARN] TrainingArguments: '{name}' not supported in transformers "
              f"{_transformers_mod.__version__} — skipped.")
        return {}

    train_kwargs: dict = {
        "output_dir":                      args.output_dir,
        "per_device_train_batch_size":     args.per_device_train_batch_size,
        "gradient_accumulation_steps":     args.gradient_accumulation_steps,
        "gradient_checkpointing":          use_gc,
        "learning_rate":                   args.learning_rate,
        "weight_decay":                    args.weight_decay,
        "warmup_steps":                    args.warmup_steps,
        "num_train_epochs":                args.num_train_epochs,
        "logging_steps":                   args.logging_steps,
        "save_steps":                      args.save_steps,
        "save_total_limit":                args.save_total_limit,
        "bf16":                            (args.dtype == "bf16"),
        "fp16":                            (args.dtype == "fp16"),
        "optim":                           optim,
        "lr_scheduler_type":               "cosine",
        "dataloader_num_workers":          args.dataloader_num_workers,
        "seed":                            args.seed,
        "report_to":                       "none",
        "remove_unused_columns":           False,
    }
    # These were removed or renamed in some transformers versions — add only if supported
    train_kwargs.update(_train_arg("tf32",            args.tf32))
    train_kwargs.update(_train_arg("group_by_length", args.group_by_length))
    train_kwargs.update(_train_arg("torch_compile",   args.torch_compile))
    train_kwargs.update(_train_arg("save_only_model", args.save_only_model))
    # Non-reentrant GC: safer with custom model architectures (Mamba-hybrid), avoids
    # potential in-place ops conflicts during the recompute pass.
    if use_gc:
        train_kwargs.update(_train_arg("gradient_checkpointing_kwargs", {"use_reentrant": False}))

    train_args = TrainingArguments(**train_kwargs)

    trainer = Trainer(
        model=model,
        args=train_args,
        train_dataset=train_ds,
        data_collator=collator,
    )

    resume = args.resume_from_checkpoint or None
    if resume:
        print(f"[*] Resuming from checkpoint: {resume}")
    print("[*] Starting training ...")
    trainer.train(resume_from_checkpoint=resume)

    print(f"[*] Saving LoRA adapter to {args.output_dir}")
    model.save_pretrained(args.output_dir)
    tokenizer.save_pretrained(args.output_dir)
    print("[+] Done.")


if __name__ == "__main__":
    main()


"""
# =============================================================================
# Installation
# =============================================================================

# With Unsloth (fast path, recommended):
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
pip install --no-deps trl peft accelerate bitsandbytes
pip install flash-attn --no-build-isolation

# Without Unsloth (standard path, works anywhere):
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
pip install transformers peft accelerate bitsandbytes datasets

# =============================================================================
# Data preparation (run these first)
# =============================================================================

cd Research/

# Step 1: build challenge-native traces
python prepare_sft_dataset.py \
  --input  outputs/all_correct_traces.json \
  --output outputs/train_reasoning.jsonl

# Step 2: merge all data sources (dedup + quality filter)
python merge_sft_datasets.py \
  --inputs outputs/train_reasoning.jsonl \
           data/distill_v2b.jsonl \
           data/distill_v2c.jsonl \
           data/aug_outv2b.jsonl \
  --output outputs/sft_combined.jsonl \
  --min_quality_score 0.5 \
  --max_distill 8000 \
  --max_aug 2000

# =============================================================================
# Training — Option A: use pre-merged combined file (recommended)
# =============================================================================

python nemotron_unsloth_mixed_sft_v3_2.py \
  --model_name nvidia/Nemotron-H-8B-Base-BF16 \
  --output_dir ./nemotron_sft_output \
  --reasoning_jsonl outputs/sft_combined.jsonl \
  --max_seq_length 4096 \
  --num_train_epochs 2 \
  --per_device_train_batch_size 4 \
  --gradient_accumulation_steps 4 \
  --learning_rate 1e-4 \
  --warmup_steps 100 \
  --load_in_4bit \
  --dtype bf16 \
  --lora_r 32 \
  --lora_alpha 64 \
  --target_preset kaggle

# =============================================================================
# Training — Option B: pass raw files directly, let trainer do mixing
# =============================================================================

python nemotron_unsloth_mixed_sft_v3_2.py \
  --model_name nvidia/Nemotron-H-8B-Base-BF16 \
  --output_dir ./nemotron_sft_output \
  --reasoning_jsonl outputs/train_reasoning.jsonl \
                    data/distill_v2b.jsonl \
                    data/aug_outv2b.jsonl \
  --min_quality_score 0.5 \
  --downweight_factor 0.4 \
  --max_seq_length 4096 \
  --num_train_epochs 2 \
  --per_device_train_batch_size 4 \
  --gradient_accumulation_steps 4 \
  --learning_rate 1e-4 \
  --warmup_steps 100 \
  --load_in_4bit \
  --dtype bf16 \
  --lora_r 32 \
  --lora_alpha 64 \
  --target_preset kaggle

# =============================================================================
# Training without Unsloth (force HF+PEFT path)
# =============================================================================

python nemotron_unsloth_mixed_sft_v3_2.py \
  --model_name nvidia/Nemotron-H-8B-Base-BF16 \
  --output_dir ./nemotron_sft_output \
  --reasoning_jsonl outputs/sft_combined.jsonl \
  --no_unsloth \
  --per_device_train_batch_size 2 \
  --gradient_accumulation_steps 8 \
  --load_in_4bit \
  --dtype bf16 \
  --lora_r 32 \
  --lora_alpha 64

# =============================================================================
# Kaggle (no-internet) — Nemotron-H 8B with mamba_ssm + local packages
# =============================================================================
#
# Environment layout assumed:
#   /kaggle/input/datasets/lkk688/unslothpackages/offline_packages/  — wheel files + unsloth_source/
#   /kaggle/input/datasets/lkk688/cleandata/sft_combined_v5.jsonl    — training data
#
# Recommended kernel cell order:
#
# Cell 1 — run setup (copies Triton, installs local packages, imports mamba_ssm):
python nemotron_unsloth_mixed_sft_v3_2.py \
  --kaggle_mode \
  --model_kaggle_id "nvidia/nemotron-h/transformers/nemotron-h-8b-bf16" \
  --local_packages_dir /kaggle/input/datasets/lkk688/unslothpackages/offline_packages \
  --output_dir /kaggle/working/nemotron_sft_v5 \
  --reasoning_jsonl /kaggle/input/datasets/lkk688/cleandata/sft_combined_v5.jsonl \
  --no_unsloth \
  --dtype bf16 \
  --lora_r 32 \
  --lora_alpha 64 \
  --target_preset kaggle \
  --per_device_train_batch_size 4 \
  --gradient_accumulation_steps 8 \
  --learning_rate 1e-4 \
  --num_train_epochs 2 \
  --max_seq_length 4096 \
  --max_reasoning_words 600 \
  --downweight_factor 0.25 \
  --min_quality_score 0.5 \
  --include_think_for_reasoning \
  --seed 3407

# For the 30B model (RTX PRO 6000 / 94 GB VRAM), no 4-bit needed.
# GC MUST be enabled (default): 30B bf16 ~60GB + LoRA optimizer ~7GB
# leaves only ~28GB for activations — gradient checkpointing is essential.
# Do NOT pass --no_gradient_checkpointing or --flash_attn (Nemotron-H Mamba-hybrid
# doesn't support FA2; the script falls back gracefully but it wastes a load attempt).
python nemotron_unsloth_mixed_sft_v3_2.py \
  --kaggle_mode \
  --model_name $MODEL_PATH \
  --output_dir /kaggle/working/nemotron_sft_v5 \
  --reasoning_jsonl /kaggle/input/datasets/lkk688/reasoning-sft-combined/sft_combined_v5.jsonl \
  --no_unsloth \
  --dtype bf16 \
  --lora_r 32 \
  --lora_alpha 64 \
  --target_preset kaggle \
  --per_device_train_batch_size 4 \
  --gradient_accumulation_steps 8 \
  --learning_rate 1e-4 \
  --num_train_epochs 2 \
  --max_seq_length 4096 \
  --max_reasoning_words 600 \
  --downweight_factor 0.25 \
  --min_quality_score 0.5 \
  --include_think_for_reasoning

# Override individual Kaggle paths if the kernel layout differs:
#   --cutlass_path  /path/to/nvidia_cutlass_dsl/python_packages/
#   --triton_src    /path/to/read-only/triton
#   --triton_dst    /kaggle/working/triton   (writable copy destination)
"""


Overwriting nemotron_unsloth_mixed_sft_v3_2.py


In [8]:
!ls

nemotron_unsloth_mixed_sft_v3_2.py  nvidia-utility-script


In [9]:
!pwd

/kaggle/working


In [13]:
import kagglehub
MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")

In [14]:
!echo $MODEL_PATH

/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1


In [17]:
!python nemotron_unsloth_mixed_sft_v3_2.py \
  --kaggle_mode \
  --model_name $MODEL_PATH \
  --model_kaggle_id "nvidia/nemotron-h/transformers/nemotron-h-8b-bf16" \
  --output_dir /kaggle/working/nemotron_sft_v5 \
  --reasoning_jsonl /kaggle/input/datasets/lkk688/reasoning-sft-combined/sft_combined_v5.jsonl \
  --no_unsloth --dtype bf16 --lora_r 32 --lora_alpha 64 \
  --target_preset kaggle --per_device_train_batch_size 4 \
  --gradient_accumulation_steps 8 --num_train_epochs 2 \
  --max_reasoning_words 600 --downweight_factor 0.25


[*] Unsloth not found — using vanilla HF + PEFT path.
/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/torch/compiler/__init__.py:148: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  return torch._dynamo.allow_in_graph(fn)
[*] Kaggle mode: setting up offline environment ...
[Kaggle] mamba_ssm path added: /kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/nvidia_cutlass_dsl/python_packages/
[Kaggle] Triton already at /kaggle/working/triton
[*] Downloading model via kagglehub: nvidia/nemotron-h/transformers/nemotron-h-8b-bf16
[WARN] kagglehub.model_download failed (POST failed with: {"errors":["Permission \u0027models.get\u0027 was denied"],"error":{"code":7},"wasSuccessful":false}) — falling back to --model_name
[*] Loading model with vanilla HF + PEFT ...
Loading weights: 100%|████████████████████| 6243/6243 [00:04<00:00, 1424.72it/s]
[*] Applying LoRA (PEFT) preset=kag

In [18]:
# Check 1: is the package installed?
!pip show flash-attn 2>/dev/null || echo "flash-attn NOT installed"



Name: flash_attn
Version: 2.8.3
Summary: Flash Attention: Fast and Memory-Efficient Exact Attention
Home-page: https://github.com/Dao-AILab/flash-attention
Author: Tri Dao
Author-email: tri@tridao.me
License: 
Location: /kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script
Requires: einops, torch
Required-by: 


In [19]:
# Check 2: scan the NVIDIA utility-script tree (Kaggle sometimes bundles it)
!find /kaggle/usr/lib/notebooks -name "flash_attn*" 2>/dev/null | head -5

/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/hopper/flash_attn_interface.py
/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/torch/include/ATen/native/transformers/cuda/flash_attn
/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/torch/include/ATen/native/transformers/hip/flash_attn
/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/flash_attn_2_cuda.cpython-312-x86_64-linux-gnu.so
/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/flash_attn-2.8.3.dist-info


In [20]:
# Check 3: transformers' own detector
from transformers.utils import is_flash_attn_2_available
print("Flash Attention 2 available:", is_flash_attn_2_available())

# Check 4: confirm nvcc exists (needed to build from source)
!nvcc --version

Flash Attention 2 available: True
nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2024 NVIDIA Corporation
Built on Thu_Jun__6_02:18:23_PDT_2024
Cuda compilation tools, release 12.5, V12.5.82
Build cuda_12.5.r12.5/compiler.34385749_0


Why Nemotron-H doesn't support FA2: It's a Mamba-hybrid architecture (~75% Mamba SSM layers + ~25% standard transformer attention layers). The Mamba layers process sequences with a selective state-space model — no Q/K/V attention at all — so Flash Attention simply isn't applicable to most of the model. The few transformer layers it does have haven't been wired to FA2's dispatch mechanism in the Kaggle-bundled modeling_nemotron_h.py

In [28]:
!nvidia-smi

Wed May  6 17:11:15 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   31C    P0             46W /  600W |       3MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [30]:
!python nemotron_unsloth_mixed_sft_v3_2.py \
  --kaggle_mode \
  --model_name $MODEL_PATH \
  --output_dir /kaggle/working/nemotron_sft_v5 \
  --reasoning_jsonl /kaggle/input/datasets/lkk688/reasoning-sft-combined/sft_combined_v5.jsonl \
  --no_unsloth --dtype bf16 \
  --lora_r 32 --lora_alpha 64 \
  --target_preset kaggle \
  --per_device_train_batch_size 4 \
  --gradient_accumulation_steps 8 \
  --num_train_epochs 2 \
  --max_reasoning_words 600 --downweight_factor 0.25



[*] Unsloth not found — using vanilla HF + PEFT path.
/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/torch/compiler/__init__.py:148: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  return torch._dynamo.allow_in_graph(fn)
[*] Kaggle mode: setting up offline environment ...
[Kaggle] nvidia-utility-script root added: /kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script
[Kaggle] cutlass DSL path added: /kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/nvidia_cutlass_dsl/python_packages/
[Kaggle] Triton already at /kaggle/working/triton
[*] Loading model with vanilla HF + PEFT ...
Loading weights: 100%|████████████████████| 6243/6243 [00:04<00:00, 1415.60it/s]
[*] Applying LoRA (PEFT) preset=kaggle
/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_tensor.py:122: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future v

In [31]:
import os, shutil

# See what's eating space
os.system("df -h /kaggle/working")
os.system("du -sh /kaggle/working/* 2>/dev/null | sort -rh | head -20")


Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G   20G     0 100% /kaggle/working
11G	/kaggle/working/nemotron_sft_v5
8.9G	/kaggle/working/nvidia-utility-script
641M	/kaggle/working/triton
36K	/kaggle/working/nemotron_unsloth_mixed_sft_v3_2.py


0

In [33]:
!ls /kaggle/working/nemotron_sft_v5/checkpoint-400

adapter_model.safetensors  README.md


In [34]:
import shutil, os

# 8.9 GB — redundant copy; original is at /kaggle/usr/lib/... (already mounted)
shutil.rmtree("/kaggle/working/nvidia-utility-script")
print("Deleted nvidia-utility-script copy")

# Corrupted partial checkpoint from the crashed save
shutil.rmtree("/kaggle/working/nemotron_sft_v5/checkpoint-400", ignore_errors=True)
print("Deleted checkpoint-400")

# Verify checkpoint-200 is intact
ckpt = "/kaggle/working/nemotron_sft_v5/checkpoint-200"
print("checkpoint-200:", os.listdir(ckpt) if os.path.exists(ckpt) else "MISSING")

os.system("df -h /kaggle/working")


Deleted nvidia-utility-script copy
Deleted checkpoint-400
checkpoint-200: ['optimizer.pt', 'tokenizer.json', 'README.md', 'rng_state.pth', 'adapter_model.safetensors', 'trainer_state.json', 'chat_template.jinja', 'tokenizer_config.json', 'adapter_config.json', 'scheduler.pt', 'training_args.bin']
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G   11G  9.1G  54% /kaggle/working


0

In [47]:
!ls /kaggle/working/nemotron_sft_v5/

checkpoint-200	checkpoint-400


In [46]:
# See what's eating space
os.system("df -h /kaggle/working")
os.system("df -h /kaggle/working/nemotron_sft_v5/")
os.system("du -sh /kaggle/working/* 2>/dev/null | sort -rh | head -20")

Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G   20G     0 100% /kaggle/working
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G   20G     0 100% /kaggle/working
19G	/kaggle/working/nemotron_sft_v5
641M	/kaggle/working/triton
64M	/kaggle/working/bin
40K	/kaggle/working/nemotron_unsloth_mixed_sft_v3_2.py


0

In [36]:
# 2. Delete the Triton copy (setup_kaggle_env will re-copy it when training restarts)
shutil.rmtree("/kaggle/working/triton", ignore_errors=True)
print("Deleted triton copy")

# 3. Verify checkpoint-200 is intact (this is the resume point)
ckpt = "/kaggle/working/nemotron_sft_v5/checkpoint-200"
print("checkpoint-200 files:", os.listdir(ckpt) if os.path.exists(ckpt) else "MISSING!")

os.system("df -h /kaggle/working")  # should have ~5-7GB free now

Deleted triton copy
checkpoint-200 files: ['optimizer.pt', 'tokenizer.json', 'README.md', 'rng_state.pth', 'adapter_model.safetensors', 'trainer_state.json', 'chat_template.jinja', 'tokenizer_config.json', 'adapter_config.json', 'scheduler.pt', 'training_args.bin']
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G  9.9G  9.7G  51% /kaggle/working


0

In [43]:
!nvidia-smi

Wed May  6 23:03:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   31C    P0             46W /  600W |       3MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [42]:
!python nemotron_unsloth_mixed_sft_v3_2.py \
  --kaggle_mode \
  --model_name $MODEL_PATH \
  --output_dir /kaggle/working/nemotron_sft_v5 \
  --reasoning_jsonl /kaggle/input/datasets/lkk688/reasoning-sft-combined/sft_combined_v5.jsonl \
  --no_unsloth --dtype bf16 \
  --lora_r 32 --lora_alpha 64 \
  --target_preset kaggle \
  --per_device_train_batch_size 4 \
  --gradient_accumulation_steps 8 \
  --num_train_epochs 2 \
  --max_reasoning_words 600 --downweight_factor 0.25 \
  --save_steps 400 \
  --save_total_limit 1 \
  --resume_from_checkpoint /kaggle/working/nemotron_sft_v5/checkpoint-200


[*] Unsloth not found — using vanilla HF + PEFT path.
/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/torch/compiler/__init__.py:148: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  return torch._dynamo.allow_in_graph(fn)
[*] Kaggle mode: setting up offline environment ...
[Kaggle] nvidia-utility-script root added: /kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script
[Kaggle] cutlass DSL path added: /kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/nvidia_cutlass_dsl/python_packages/
[Kaggle] ptxas binary → /kaggle/working/bin/ptxas-blackwell
[Kaggle] TRITON_PTXAS_BLACKWELL_PATH=/kaggle/working/bin/ptxas-blackwell
[Kaggle] ptxas binary → /kaggle/working/bin/ptxas
[Kaggle] TRITON_PTXAS_PATH=/kaggle/working/bin/ptxas
[Kaggle] Triton already at /kaggle/working/triton
[*] Loading model with vanilla HF + PEFT ...
Loading weights: 100%|████████████████████| 6243/624

In [49]:
import os, shutil

shutil.rmtree("/kaggle/working/nemotron_sft_v5/checkpoint-400", ignore_errors=True)
print("Deleted checkpoint-400")
os.system("df -h /kaggle/working && du -sh /kaggle/working/*")


Deleted checkpoint-400
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G   11G  9.0G  55% /kaggle/working
64M	/kaggle/working/bin
9.9G	/kaggle/working/nemotron_sft_v5
40K	/kaggle/working/nemotron_unsloth_mixed_sft_v3_2.py
641M	/kaggle/working/triton


0

The root cause is now clear: the optimizer state for adamw_torch_fused is ~6 GB (883M LoRA params × 2 moments × 4 bytes). With save_total_limit=1, saving checkpoint-400 while checkpoint-200 still exists requires 12 GB for checkpoints alone — it fits briefly but corrupts mid-write.

Fix: --save_only_model — saves only the 300 MB adapter weights, skips the 6 GB optimizer state. With --save_only_model, the disk sequence at step 400 is:

Write adapter only → 300 MB
Delete checkpoint-200 → free 6.3 GB
Net change: +6 GB free
The trade-off is you can't resume from these new checkpoints (no optimizer state).

In [ ]:
!python nemotron_unsloth_mixed_sft_v3_2.py \
  --kaggle_mode \
  --model_name $MODEL_PATH \
  --output_dir /kaggle/working/nemotron_sft_v5 \
  --reasoning_jsonl /kaggle/input/datasets/lkk688/reasoning-sft-combined/sft_combined_v5.jsonl \
  --no_unsloth --dtype bf16 \
  --lora_r 32 --lora_alpha 64 \
  --target_preset kaggle \
  --per_device_train_batch_size 4 \
  --gradient_accumulation_steps 8 \
  --num_train_epochs 2 \
  --max_reasoning_words 600 --downweight_factor 0.25 \
  --save_steps 400 \
  --save_total_limit 1 \
  --save_only_model \
  --resume_from_checkpoint /kaggle/working/nemotron_sft_v5/checkpoint-200


[*] Unsloth not found — using vanilla HF + PEFT path.
/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/torch/compiler/__init__.py:148: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  return torch._dynamo.allow_in_graph(fn)
[*] Kaggle mode: setting up offline environment ...
[Kaggle] nvidia-utility-script root added: /kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script
[Kaggle] cutlass DSL path added: /kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/nvidia_cutlass_dsl/python_packages/
[Kaggle] TRITON_PTXAS_BLACKWELL_PATH=/kaggle/working/bin/ptxas-blackwell
[Kaggle] TRITON_PTXAS_PATH=/kaggle/working/bin/ptxas
[Kaggle] Triton already at /kaggle/working/triton
[*] Loading model with vanilla HF + PEFT ...
Loading weights: 100%|████████████████████| 6243/6243 [00:04<00:00, 1431.53it/s]
[*] Applying LoRA (PEFT) preset=kaggle
/usr/local/lib/python3.12/dist-packages/to

In [ ]:
import subprocess, json

# Verify checkpoint-400 is complete (save_only_model = adapter only, ~300MB)
import os
ckpt = "/kaggle/working/nemotron_sft_v5/checkpoint-600"
print("checkpoint-600 files:", os.listdir(ckpt))

# Check session time vs training time
state = json.load(open(f"{ckpt}/trainer_state.json"))
print(f"Global step: {state['global_step']}, epoch: {state['epoch']:.3f}")
print(f"Output disk: ", end=""); os.system("df -h /kaggle/working | tail -1")


In [ ]:
# Push checkpoint to a Kaggle dataset for later use
subprocess.run([
    "kaggle", "datasets", "create",
    "--path", "/kaggle/working/nemotron_sft_v5/checkpoint-600",
    "--name", "nemotron-lora-sft-v5-ckpt600",
    "--title", "Nemotron LoRA SFT v5 checkpoint-600"
], check=False)


In [ ]:
import site

cutlass_pkg_path = "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/nvidia_cutlass_dsl/python_packages/"
site.addsitedir(cutlass_pkg_path)

import kagglehub
import mamba_ssm
import torch
from peft import LoraConfig, get_peft_model, get_peft_model_state_dict, TaskType
from transformers import AutoModelForCausalLM, AutoTokenizer

# Configuration
MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
OUTPUT_DIR = "/kaggle/working"
LORA_RANK = 32  # Can be set to a maximum of 32

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map="auto",
    trust_remote_code=True,
    dtype=torch.bfloat16
)
# tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
print("Model loaded successfully.")

# Initialize LoRA Adapter
print(f"Initializing LoRA adapter with rank={LORA_RANK}...")
lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=16,
    target_modules=r".*\.(in_proj|out_proj|up_proj|down_proj)$",
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

# Apply LoRA to the model
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


# YOUR CODE HERE
# --------------
# model.train() 
# --------------


# Save Adapter
print(f"Saving adapter to {OUTPUT_DIR}...")
model.save_pretrained(OUTPUT_DIR)

In [ ]:

print("\n" + "="*50)
print("🔍 MODEL ARCHITECTURE:")
print(model)
print("="*50 + "\n")

In [ ]:

print("\n🔍 VERIFYING LORA TARGET MODULES:")
trainable_layers = []
for name, module in model.named_modules():
    if "lora" in name.lower() or getattr(module, "requires_grad", False):
        # 
        if hasattr(module, "weight") and module.weight.requires_grad:
            trainable_layers.append(name)
            
print(f"[+] Found {len(trainable_layers)} trainable parameter tensors.")

for layer in trainable_layers[:10]:
    print(f"    - {layer}")
print("="*50 + "\n")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
#
if getattr(tokenizer, "chat_template", None) is None:
    print("[*] Injecting basic ChatML template...")
    tokenizer.chat_template = "{% for message in messages %}{{'<|im_start|>' + message['role'] + '\n' + message['content'] + '<|im_end|>\n'}}{% endfor %}{% if add_generation_prompt %}{{ '<|im_start|>assistant\n' }}{% endif %}"

In [ ]:
import subprocess

subprocess.run("zip -m submission.zip *", shell=True, check=True)

In [ ]:
print('Done.')